<a href="https://colab.research.google.com/github/RashiBista/practice1/blob/main/Scrapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install once
!pip install playwright beautifulsoup4 pandas
!playwright install chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 MB 19.2 MB/s eta 0:00:00
175.4 MiB [] 0% 0.0s175.4 MiB [] 0% 118.1s175.4 MiB [] 0% 400.8s175.4 MiB [] 0% 453.7s175.4 MiB [] 0% 364.1s175.4 MiB [] 0% 315.5s175.4 MiB [] 0% 283.6s175.4 MiB [] 0% 261.5s175.4 MiB [] 0% 244.5s175.4 MiB [] 0% 228.9s175.4 MiB [] 0% 215.4s175.4 MiB [] 0% 203.4s175.4 MiB [] 0% 178.4s175.4 MiB [] 0% 162.3s175.4 MiB [] 0% 146.2s175.4 MiB [] 0% 139.6s175.4 MiB [] 0% 130.6s175.4 MiB [] 0% 120.3s175.4 MiB [] 0% 109.8s175.4 MiB [] 0% 99.6s175.4 MiB [] 0% 89.6s175.4 MiB [] 0% 81.2s175.4 MiB [] 0% 75.6s175.4 MiB [] 0% 69.4s175.4 MiB [] 0% 63.8s175.4 MiB [] 1% 58.4s175.4 MiB [] 1% 52.6s175.4 MiB [] 1% 46.8s175.4 MiB [] 1% 42.6s175.4 MiB [] 1% 38.6s175.4 MiB [] 1% 37.6s175.4 MiB [] 1% 37.9s175.4 MiB [] 2% 37.8s175.4 MiB [] 2% 37.5s175.4 MiB [] 2% 37.1s175.4 MiB [] 2% 37.0s175.4 MiB [] 2% 35.9s175.4 MiB [] 2% 30.5s175.4 MiB [] 3% 26.6s175.4 MiB [] 3% 22.3s175.4 MiB [] 4% 21.0s175.4 MiB [] 4% 18.6s175.4 MiB [] 5% 17.3s1

In [ ]:
# Install necessary system dependencies for Playwright Chromium
!apt-get update
!apt-get install -y libatk1.0-0 libgdk-pixbuf2.0-0 libgtk-3-0 libgbm-dev libasound2 libnss3 libxss1

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,303 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,183 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,006 kB]
Get:13 https://cli.github.com/packages stable/main amd64 Packages [355 B

In [ ]:
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import pandas as pd

async def scrape_dda_notices():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        await page.goto("https://dda.gov.np/visual-list/",
                       wait_until="networkidle")
        content = await page.content()
        await browser.close()

        soup = BeautifulSoup(content, "html.parser")
        notices = []

        # Extract all notice headings and links
        for heading in soup.select("h3 a"):
            text = heading.text.strip()
            link = "https://dda.gov.np" + heading.get("href", "")

            # Flag veterinary-related notices
            vet_keywords = ["पशु", "veterinary", "animal", "vet"]
            is_vet = any(kw.lower() in text.lower() for kw in vet_keywords)

            notices.append({
                "title": text,
                "url": link,
                "is_veterinary": is_vet
            })

        df = pd.DataFrame(notices)
        df.to_csv("dda_notices.csv", index=False)
        print(f"Scraped {len(df)} notices, {df['is_veterinary'].sum()} veterinary")
        return df

await scrape_dda_notices()

Scraped 86 notices, 5 veterinary


,title,url,is_veterinary
0,पशुपंक्षी औषधि तालिम लिएका व्यक्तिहरुको परिक्ष...,https://dda.gov.np/content/235/regarding-the-c...,True
1,"Pregabalin, Dicyclomine र Promethazine काे उत्...",https://dda.gov.np/content/233/notification-re...,False
2,क्यान्सरमा प्रयोग हुने औषधिहरुको उपलब्धता सम्ब...,https://dda.gov.np/content/232/availability-of...,False
3,नयाँ औषधि तथा समिश्रणहरुको मुल्यांकन फाराम संश...,https://dda.gov.np/content/230/revision-of-eva...,False
4,क्यान्सरमा प्रयोग हुने प्ल्याटिनम ग्रुपका औषधि...,https://dda.gov.np/content/225/urgent-notice-r...,False
...,...,...,...
81,नयाँ औषधि तथा समिश्रणहरुको मुल्यांकन फाराम संश...,https://dda.gov.np/content/230/revision-of-eva...,False
82,क्यान्सरमा प्रयोग हुने प्ल्याटिनम ग्रुपका औषधि...,https://dda.gov.np/content/225/urgent-notice-r...,False
83,How to Use a Pressurized Dose Inhaler Correctl...,https://dda.gov.nphttps://youtu.be/vgd3K-Zcoyg,False
84,How to Use a Rotahaler Properly | रोटाहेलर सही...,https://dda.gov.nphttps://youtu.be/pn9PV-iOenw,False


In [ ]:
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import pandas as pd

async def scrape_dams_veterinary_data():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )
        page = await context.new_page()

        target_url = "https://dams.dda.gov.np/manLogin/publicInfo"
        print(f"Navigating to: {target_url}")

        try:
            await page.goto(target_url, wait_until="networkidle", timeout=60000)
        except Exception as e:
            print(f"Initial load timed out, retrying with domecontentloaded... Error: {e}")
            await page.goto(target_url, wait_until="domcontentloaded")

        # =======================================================================
        # 👇 INSERTED FIX: SIMULATE HUMAN SEARCH SELECTION TO LOAD THE TABLE 👇
        # =======================================================================
        print("Waiting for search selection fields to load...")
        try:
            # 1. Wait for the interactive dropdown or input field element to appear
            await page.wait_for_selector(".select2-selection, select, input[type='search'], input", timeout=15000)

            # 2. Click the dropdown container to open the search element
            # Note: DAMS uses Select2 components widely; clicking input or container triggers it
            await page.click(".select2-selection, input")
            await page.wait_for_timeout(500)

            # 3. Type a common veterinary API term to populate general records
            # Typing 'AMOXICILLIN' or hitting 'Enter' safely requests a default or matched query matrix
            await page.keyboard.type("AMOXICILLIN")
            await page.keyboard.press("Enter")
            print("Search query submitted. Waiting for data grid rows to populate...")

            # 4. Give the server time to return the rows via AJAX
            await page.wait_for_selector("table tr td", timeout=15000)
            print("Data table detected! Starting extraction loop.")

        except Exception as interaction_error:
            print(f"Interaction warning: {interaction_error}. Proceeding to fallback check.")
            # Give a generic fallback wait time if selectors differ slightly across updates
            await page.wait_for_timeout(5000)
        # =======================================================================
        # 👆 END OF INSERTED FIX 👆
        # =======================================================================

        all_records = []
        current_page = 1
        max_pages_to_scrape = 10

        while current_page <= max_pages_to_scrape:
            print(f"--- Processing Table Data Grid: Page {current_page} ---")

            html_content = await page.content()
            soup = BeautifulSoup(html_content, "html.parser")

            target_table = soup.find("table")

            if not target_table:
                print("No standard data matrix table structure discovered on this page view.")
                break

            rows = target_table.find_all("tr")
            start_index = 1 if rows[0].find("th") or "brand" in rows[0].text.lower() else 0

            page_row_count = 0
            for row in rows[start_index:]:
                cols = [td.text.strip() for td in row.find_all("td")]

                if len(cols) >= 3:
                    page_row_count += 1

                    brand = cols[0]
                    generic = cols[1] if len(cols) > 1 else ""
                    strength = cols[2] if len(cols) > 2 else ""
                    manufacturer = cols[3] if len(cols) > 3 else "Unknown"

                    is_vet = any(word in generic.lower() or word in brand.lower()
                                 for word in ["vet", "veterinary", "animal", "poultry", "bovine"])

                    all_records.append({
                        "brand_name": brand,
                        "generic_composition": generic,
                        "strength": strength,
                        "manufacturer_or_importer": manufacturer,
                        "is_veterinary_target": is_vet
                    })

            print(f"Extracted {page_row_count} drug records from page grid.")

            next_anchor = await page.query_selector("a:has-text('Next'), li.next a, button:has-text('Next')")

            if next_anchor:
                parent_element = await next_anchor.evaluate_handle("el => el.parentElement")
                parent_class = await parent_element.evaluate("el => el.className") if parent_element else ""

                if "disabled" in parent_class or not await next_anchor.is_enabled():
                    print("Reached final page boundary. Next interaction controller disabled.")
                    break

                print("Triggering interactive click event on pagination element...")
                await next_anchor.click()

                await page.wait_for_timeout(2500)
                current_page += 1
            else:
                print("No identifiable pagination sequence controllers found on current node template.")
                break

        await browser.close()

        df = pd.DataFrame(all_records)
        if not df.empty:
            df.to_csv("dams_extracted_pharmaceuticals.csv", index=False)
            print(f"\n Scraping task finalized! Saved {len(df)} structural database seeds into 'dams_extracted_pharmaceuticals.csv'.")
            print(df.head(10))
        else:
            print("\n Execution completed, but zero records met parsing index constraints.")

        return df

await scrape_dams_veterinary_data()

Navigating to: https://dams.dda.gov.np/manLogin/publicInfo
Waiting for search selection fields to load...
Interaction warning: Page.wait_for_selector: Timeout 15000ms exceeded.
Call log:
  - waiting for locator(".select2-selection, select, input[type='search'], input") to be visible
    29 × locator resolved to 6 elements. Proceeding with the first one: <input type="text" role="textbox" autocomplete="off" aria-label="Search" class="form-control"/>
. Proceeding to fallback check.
--- Processing Table Data Grid: Page 1 ---
No standard data matrix table structure discovered on this page view.

 Execution completed, but zero records met parsing index constraints.


""


In [ ]:
import urllib.request
import pandas as pd

# The official PDF/Data endpoint for Nepal's Approved Veterinary Drugs
url = "https://vsdrl.gov.np/downloadsfiles/National-Essential-Veterinary-Medicine-List-1762523940.pdf"
output_pdf = "National_Essential_Veterinary_Medicine_List_Nepal.pdf"

print("Downloading official Nepalese Veterinary Medicine Dataset...")
try:
    urllib.request.urlretrieve(url, output_pdf)
    print(f"Successfully downloaded to {output_pdf}!")
except Exception as e:
    print(f"Download failed: {e}. You can manually download from vsdrl.gov.np and upload to Colab.")

Download failed: HTTP Error 406: Not Acceptable. You can manually download from vsdrl.gov.np and upload to Colab.


In [ ]:
# First install pdfplumber inside Colab
!pip install pdfplumber -q

import pdfplumber
import pandas as pd
import urllib.request

pdf_path = "National_Essential_Veterinary_Medicine_List_Nepal.pdf"
url = "https://vsdrl.gov.np/downloadsfiles/National-Essential-Veterinary-Medicine-List-1762523940.pdf"

print("Downloading official Nepalese Veterinary Medicine Dataset...")
try:
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as response, open(pdf_path, 'wb') as out_file:
        out_file.write(response.read())
    print(f"Successfully downloaded to {pdf_path}!")
except Exception as e:
    print(f"Download failed: {e}. Please ensure the URL is correct or try manually downloading.")
    raise

extracted_rows = []

print("Extracting tables directly from official document...")
try:
    with pdfplumber.open(pdf_path) as pdf:
        # Print raw text of the first page to diagnose text extraction issues
        if pdf.pages:
            first_page_text = pdf.pages[0].extract_text()
            print("\n--- Raw text from PDF Page 1 ---")
            print(first_page_text)
            print("----------------------------------")

        for page_num, page in enumerate(pdf.pages, start=1):
            table = page.extract_table()
            if table:
                # Debugging: Print the extracted table structure (with content)
                # print(f"\n--- Table extracted from Page {page_num} ---")
                # for r in table: print(r)
                # print("----------------------------------")

                for row in table:
                    cleaned_row = [str(cell).replace('\n', ' ').strip() if cell else "" for cell in row]

                    # The original condition `len(cleaned_row) >= 4` might be too strict or column detection is off
                    # Let's adjust to be less strict for debugging, and check for actual content
                    if any(cell_content for cell_content in cleaned_row) and cleaned_row[0] != "S.N.": # Check if row has any content
                        extracted_rows.append(cleaned_row)

    df_vet = pd.DataFrame(extracted_rows)
    if not df_vet.empty:
        # Assuming the first row is a header, set it and clean data
        # This part might need further refinement based on actual text output
        header = df_vet.iloc[0]
        df_vet = df_vet[1:]
        df_vet.columns = header
        df_vet = df_vet.dropna(how='all') # Drop rows that might be entirely empty after header removal

    df_vet.to_csv("nepal_essential_veterinary_drugs.csv", index=False)

    print(f"Done! Extracted {len(df_vet)} clean veterinary drug formulas.")
    print("Preview of your clean database seed rows:")
    print(df_vet.head(10))
except FileNotFoundError:
    print(f"Error: PDF file '{pdf_path}' not found after download attempt. This should not happen if the download was successful. Please check the URL or manual download.")
except Exception as e:
    print(f"An error occurred during PDF extraction: {e}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 96.7 MB/s eta 0:00:00
Successfully downloaded to National_Essential_Veterinary_Medicine_List_Nepal.pdf!
Extracting tables directly from official document...

--- Raw text from PDF Page 1 ---

----------------------------------
Done! Extracted 0 clean veterinary drug formulas.
Preview of your clean database seed rows:
Empty DataFrame
Columns: []
Index: []


In [ ]:
# Install Tesseract OCR and PyTesseract
!apt-get install tesseract-ocr -y
!pip install pytesseract

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 30 not upgraded.
